<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/05-agents/02-tool-design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool Design

**Goal:** Design tools the model can actually use well.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client, the only dependency this notebook needs.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

## Tool quality beats prompt tuning

When an agent misbehaves, the reflex is to edit the system prompt. Usually the actual problem is the tools. Two framings to hold onto:

- **A tool description is a prompt.** It's injected into the model's context on every single request, and it's the only documentation the model will ever read about your tool.
- **A tool interface is an API whose caller is a model.** Everything you know about API design still applies, except your consumer re-reads the docs on every call, can't step through your code, and takes error messages literally.

This notebook demonstrates that with four before/after pairs, running the same loop from notebook 01 on the same task with a *bad* tool and a *good* one. The only thing that changes between runs is the tool.

## The loop (compact copy from notebook 01)

Same loop, parameterized by a tool list and an implementation dict, and it now returns total token usage. We'll need that to put a price on bad design in pair 3.

In [ ]:
import json

def run_agent(task, tools, impls, max_turns=8, verbose=True):
    """The notebook-01 loop, compact. Returns (final_text, usage_totals)."""
    messages = [{"role": "user", "content": task}]
    usage = {"input_tokens": 0, "output_tokens": 0}

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages,
        )
        usage["input_tokens"] += response.usage.prompt_tokens
        usage["output_tokens"] += response.usage.completion_tokens
        choice = response.choices[0]
        msg = choice.message

        if choice.finish_reason != "tool_calls":
            text = msg.content or ""
            if verbose:
                print(f"[done in {turn} turn(s)] {text.strip()[:300]}")
                print(f"[usage] {usage}")
            return text, usage

        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = impls.get(tc.function.name)
            try:
                out = str(fn(**args)) if fn else f"Error: unknown tool '{tc.function.name}'."
            except TypeError as e:
                out = f"Error: bad arguments: {e}"
            if verbose:
                print(f"[turn {turn}] {tc.function.name}({tc.function.arguments[:90]}) -> {out.strip()[:90]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

    if verbose:
        print(f"[stopped: hit max_turns={max_turns}]")
    return "(hit max_turns)", usage

# Shared fake data for every example: a tiny order system.
ORDERS = {
    "ORD-4521": {"status": "shipped", "carrier": "UPS", "eta": "2026-08-15",
                 "customer_id": "C-102", "placed": "2026-08-08"},
    "ORD-4488": {"status": "delivered", "carrier": "FedEx", "eta": "2026-07-30",
                 "customer_id": "C-102", "placed": "2026-07-25"},
    "ORD-4530": {"status": "processing", "carrier": None, "eta": None,
                 "customer_id": "C-217", "placed": "2026-08-11"},
}
CUSTOMERS = {
    "C-102": {"name": "Dana Wong", "email": "dana@example.com", "orders": ["ORD-4488", "ORD-4521"]},
    "C-217": {"name": "Sam Patel", "email": "sam@example.com", "orders": ["ORD-4530"]},
}


## Pair 1: the description is a prompt

Both versions below wrap the *same* lookup function. The bad one describes what the tool does in four words. The good one states **when to use it, when NOT to, the exact input format, and what comes back**: the four things a model needs to make a correct call on the first try.

In [ ]:
def lookup_order(order_id):
    order = ORDERS.get(order_id)
    if order is None:
        return f"No order found for id '{order_id}'."
    return (f"status={order['status']} carrier={order['carrier']} "
            f"eta={order['eta']} placed={order['placed']}")

BAD_DESC_TOOL = [{
    "type": "function",
    "function": {
        "name": "query",
        "description": "Query the system.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
}]

GOOD_DESC_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_order_status",
        "description": (
            "Look up the shipping status of a single order. Use this when the user asks "
            "where an order is or when it will arrive. Do NOT use it for refunds, "
            "inventory, or customer profiles. order_id must be the full ID in the form "
            "'ORD-' plus digits, e.g. 'ORD-4521'. Returns status, carrier, ETA, and the "
            "date the order was placed."
        ),
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string",
                                        "description": "Full order ID, e.g. 'ORD-4521'."}},
            "required": ["order_id"],
        },
    },
}]

TASK_1 = "Where is order 4521 and when will it arrive?"

print("--- bad description ---")
run_agent(TASK_1, BAD_DESC_TOOL, {"query": lookup_order})

print("\n--- good description ---")
run_agent(TASK_1, GOOD_DESC_TOOL, {"get_order_status": lookup_order})


Run this and compare the traces. With the vague tool, the model has to *guess* the ID format. It will often pass `4521`, get "no order found", and burn a turn (or two) probing variants; sometimes it half-guesses and answers with hedged uncertainty. With the good description, the format example in the description means the first call is usually `ORD-4521` and the run finishes in two turns.

Nothing about the implementation changed. The description alone bought correctness *and* fewer paid turns.

## Pair 2: one job per tool

The mega-tool below is a real anti-pattern: one entry point, an `action` string, and eight optional parameters whose validity depends on the action. Every call is a chance to pick the wrong combination. The alternative is three focused tools where the schema itself makes invalid calls impossible.

In [ ]:
def customer_data(action=None, customer_id=None, order_id=None, field=None,
                  value=None, query=None, limit=None, sort=None, format=None):
    # One function, many failure modes -- exactly what the model experiences.
    if action == "get_customer":
        if not customer_id:
            return "Error: action 'get_customer' requires customer_id."
        c = CUSTOMERS.get(customer_id)
        return str(c) if c else f"Error: no customer '{customer_id}'."
    if action == "list_orders":
        if not customer_id:
            return "Error: action 'list_orders' requires customer_id."
        c = CUSTOMERS.get(customer_id)
        if not c:
            return f"Error: no customer '{customer_id}'."
        return ", ".join(c["orders"])
    if action == "get_order":
        if not order_id:
            return "Error: action 'get_order' requires order_id."
        o = ORDERS.get(order_id)
        return str(o) if o else f"Error: no order '{order_id}'."
    return f"Error: unknown action '{action}'. Valid: get_customer, list_orders, get_order."

MEGA_TOOL = [{
    "type": "function",
    "function": {
        "name": "customer_data",
        "description": "Access customer and order data. Set action and the relevant parameters.",
        "parameters": {
            "type": "object",
            "properties": {
                "action": {"type": "string"},
                "customer_id": {"type": "string"},
                "order_id": {"type": "string"},
                "field": {"type": "string"},
                "value": {"type": "string"},
                "query": {"type": "string"},
                "limit": {"type": "integer"},
                "sort": {"type": "string"},
            },
            "required": [],
        },
    },
}]

def get_customer(customer_id):
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"Error: no customer '{customer_id}'."
    return f"name={c['name']} email={c['email']}"

def list_orders(customer_id):
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"Error: no customer '{customer_id}'."
    lines = [f"{oid}: placed {ORDERS[oid]['placed']}" for oid in c["orders"]]
    return "\n".join(lines)

def get_order(order_id):
    o = ORDERS.get(order_id)
    if not o:
        return f"Error: no order '{order_id}'."
    return f"status={o['status']} carrier={o['carrier']} eta={o['eta']}"

def _focused(name, description, props, required):
    return {"type": "function",
            "function": {"name": name, "description": description,
                         "parameters": {"type": "object", "properties": props,
                                        "required": required}}}

FOCUSED_TOOLS = [
    _focused("get_customer",
             "Get a customer's name and email by customer_id (e.g. 'C-102').",
             {"customer_id": {"type": "string"}}, ["customer_id"]),
    _focused("list_orders",
             "List a customer's order IDs with the date each was placed, "
             "newest and oldest included. Use get_order for details on one order.",
             {"customer_id": {"type": "string"}}, ["customer_id"]),
    _focused("get_order",
             "Get status, carrier, and ETA for one order by full order ID (e.g. 'ORD-4521').",
             {"order_id": {"type": "string"}}, ["order_id"]),
]

TASK_2 = "Has customer C-102's most recent order shipped yet?"

print("--- mega-tool ---")
run_agent(TASK_2, MEGA_TOOL, {"customer_data": customer_data})

print("\n--- three focused tools ---")
run_agent(TASK_2, FOCUSED_TOOLS,
          {"get_customer": get_customer, "list_orders": list_orders, "get_order": get_order})


Run this and note the shape of each trace. The mega-tool run usually works *eventually* (models are good at recovering) but watch for wasted turns: a call with a missing parameter, an invented `action` name, a retry after an error string. The focused version tends to be a clean `list_orders` → `get_order` chain, because each schema only permits sensible calls.

Rule of thumb: if you're writing `if action == ...` dispatch inside one tool, the actions want to be separate tools. Enums and required fields are guardrails the model can't walk past; a docstring convention is not.

## Pair 3: return only what the next decision needs

This one is about money as much as accuracy. A tool result doesn't get read once. It lands in the conversation and is **re-sent with every subsequent request** in the loop. A 5,000-token JSON dump returned on turn 2 of a 6-turn run gets billed as input roughly four more times.

Below, the raw tool returns the entire order record: 40 line items, an audit log, internal flags. The distilled tool returns the four fields the task actually needs.

In [ ]:
# Build one deliberately bloated order record.
BIG_ORDER = {
    "order_id": "ORD-4521",
    "status": "shipped",
    "carrier": "UPS",
    "eta": "2026-08-15",
    "internal_flags": {"fraud_score": 0.02, "warehouse": "EWR-4", "priority_class": "B2",
                       "reprice_pending": False, "legacy_migration_id": "LM-889231"},
    "line_items": [
        {"sku": f"SKU-{i:04d}", "description": f"Replacement part, model X-{i}, rev {i % 7}",
         "qty": (i % 3) + 1, "unit_price_cents": 950 + 137 * i,
         "tax_code": "TX-STD", "fulfillment_node": "EWR-4"}
        for i in range(40)
    ],
    "audit_log": [
        {"ts": f"2026-08-08T{h:02d}:14:03Z", "actor": "system",
         "event": "state_transition", "detail": f"step_{h} completed, queue depth {h * 3}"}
        for h in range(20)
    ],
}
BIG_ORDER["total_cents"] = sum(li["qty"] * li["unit_price_cents"] for li in BIG_ORDER["line_items"])

def get_order_raw(order_id):
    if order_id != "ORD-4521":
        return f"Error: no order '{order_id}'."
    return json.dumps(BIG_ORDER)  # the full dump, every time

def get_order_summary(order_id):
    if order_id != "ORD-4521":
        return f"Error: no order '{order_id}'."
    o = BIG_ORDER
    return (f"order_id={o['order_id']} status={o['status']} eta={o['eta']} "
            f"item_count={len(o['line_items'])} total=${o['total_cents'] / 100:.2f}")

RAW_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Get an order record by full order ID (e.g. 'ORD-4521').",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]},
    },
}]
SUMMARY_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Get an order's status, ETA, item count, and total value by full "
                       "order ID (e.g. 'ORD-4521').",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]},
    },
}]

TASK_3 = "What's the total value of order ORD-4521, and has it shipped?"

print("--- raw dump ---")
_, usage_raw = run_agent(TASK_3, RAW_TOOL, {"get_order": get_order_raw})

print("\n--- distilled ---")
_, usage_distilled = run_agent(TASK_3, SUMMARY_TOOL, {"get_order": get_order_summary})

print("\n--- token bill ---")
print(f"raw:       {usage_raw}")
print(f"distilled: {usage_distilled}")


Run it and compare the two `usage` lines. Both versions get the right answer; the total is precomputed in the record, so the raw version doesn't even need to do math. But the raw run's input tokens are typically several times the distilled run's, from a *single* verbose result on a *short* task. Now scale that: a 20-turn agent whose search tool returns raw API payloads pays for those payloads on every one of the remaining turns. Verbose returns compound.

There's an accuracy cost too, not just a token cost: the answer to "has it shipped?" is one field buried in 40 line items of noise. Distillation is doing the model's attention a favor.

The design move: decide what the *next decision* needs and return exactly that. If some callers genuinely need line items, that's a second tool (`get_order_items`) or a `fields` enum, not a default firehose.

## Pair 4: errors that teach

Tools fail. The question is what the model can do with the failure. `"Error"` is a dead end; an error that says **what went wrong and what to try instead** is a course correction. Here both tools have the *same minimal description* (only the error string differs) and the task uses an ID in the wrong format, so the first call is guaranteed to fail.

In [ ]:
def get_order_bad_errors(order_id):
    o = ORDERS.get(order_id)
    if not o:
        return "Error"
    return f"status={o['status']} carrier={o['carrier']} eta={o['eta']}"

def get_order_good_errors(order_id):
    o = ORDERS.get(order_id)
    if not o:
        digits = "".join(ch for ch in order_id if ch.isdigit())
        hint = f" Did you mean 'ORD-{digits}'?" if digits and f"ORD-{digits}" in ORDERS else ""
        return (f"Error: no order with id '{order_id}'. Order IDs are 'ORD-' followed "
                f"by digits, e.g. 'ORD-4521'.{hint} Call get_order again with the full ID.")
    return f"status={o['status']} carrier={o['carrier']} eta={o['eta']}"

# Identical, deliberately minimal description for both -- the error message is
# the only variable in this experiment.
MINIMAL_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Look up an order by ID.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]},
    },
}]

TASK_4 = "Where is order 4521?"

print("--- error says 'Error' ---")
run_agent(TASK_4, MINIMAL_TOOL, {"get_order": get_order_bad_errors})

print("\n--- error teaches ---")
run_agent(TASK_4, MINIMAL_TOOL, {"get_order": get_order_good_errors})


Run it and watch the second turn of each trace. With the bare `"Error"`, the model is blind: it may retry the same call, guess at formats, or give up and tell the user the order can't be found. With the teaching error, the very next call is `ORD-4521` and the task completes, because the tool told the model how to self-correct.

Good error messages are cheap to write and pay off on every failure, forever. Include three things: what was received, why it's invalid, and a concrete next step (ideally with an example).

## The rules, distilled

| Rule | Bad | Good |
|---|---|---|
| **Name says what it does** | `query`, `process`, `handle_data` | `get_order_status`, `list_orders` |
| **Description = when to use, when not, input format, what returns** | "Query the system." | "Use when the user asks where an order is. Not for refunds. order_id like 'ORD-4521'. Returns status, carrier, ETA." |
| **One job per tool** | one tool, `action` param, 8 optional fields | three tools with 1–2 required params each |
| **Typed, constrained params** | `options: string` parsed by convention | `required` fields, `enum` for fixed choices, per-property descriptions |
| **Return what the next decision needs** | full JSON dump, audit logs and all | the 3–5 fields the task uses; separate tool for detail |
| **Errors teach** | `"Error"` | what was received, why it failed, what to try instead |

### The same symmetry, everywhere

If you build MCP servers or expose internal APIs to agents, this is the entire job: you're doing API design for a consumer that reads the documentation on **every single call** and never learns from experience across sessions. That inverts some habits (docs aren't a nice-to-have consulted once, they're the hot path) but it also means every improvement to a description or error message ships instantly to every caller. There is no client code to update.

## Exercises

1. **Add an enum guardrail.** Give `list_orders` a `sort` parameter as `{"type": "string", "enum": ["newest_first", "oldest_first"]}` and implement it. Then try the same feature as a free string with the convention buried in the description: run both five times with "show me Dana's orders, oldest first" and count format errors.
2. **Fix the mega-tool without splitting it.** Keep `customer_data` as one tool but make `action` an enum, document which params each action requires in the description, and make every error message name the missing parameter. Re-run TASK_2. How close does a well-specified mega-tool get to the focused version?
3. **Build a `fields` selector.** Change `get_order` to accept an optional `fields: array of enum` and return only those fields. Verify with the usage counters that asking for `["status"]` on the bloated order costs a fraction of the dump.
4. **Write the teaching-error kit.** Take the notebook-01 filesystem tools and upgrade every failure path: `read_file` on a missing path should suggest the closest existing path (try `difflib.get_close_matches`), `calculator` on a bad expression should show which character broke the rules. Re-run the notebook-01 task with a typo planted in it.